# VAE — Вариационный Автоэнкодер
Генерация новых изображений

## Отличие VAE от обычного Autoencoder
```
Autoencoder:        Вход → [Encoder] → z (вектор)     → [Decoder] → Выход
                                        точка

VAE:                Вход → [Encoder] → μ, σ           → [Decoder] → Выход  
                                        ↓
                                   z = μ + σ × ε
                                   (распределение!)

VAE кодирует не в точку, а в РАСПРЕДЕЛЕНИЕ (mean, std)
Это делает латентное пространство непрерывным → лучше генерация
```

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print('TensorFlow:', tf.__version__)

## 1. Загрузка данных

In [ ]:
# Загружаем MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Нормализация и reshape
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print('Train:', x_train.shape)
print('Test:', x_test.shape)

## 2. Слой семплирования (Reparametrization Trick)

In [ ]:
class Sampling(layers.Layer):
    """Семплирование z = mean + std * epsilon"""
    
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        
        # Случайный шум из N(0, 1)
        epsilon = tf.random.normal(shape=(batch, dim))
        
        # z = mean + std * epsilon (reparametrization trick)
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

print('Sampling layer создан')
print('\nReparametrization trick позволяет backprop через случайность!')

## 3. Создание VAE

In [ ]:
# Параметры
LATENT_DIM = 2  # 2D для визуализации (обычно 16-128)

# === ENCODER ===
encoder_inputs = layers.Input(shape=(28, 28, 1), name='encoder_input')

x = layers.Conv2D(32, 3, activation='relu', strides=2, padding='same')(encoder_inputs)
x = layers.Conv2D(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Flatten()(x)
x = layers.Dense(256, activation='relu')(x)

# Выходы: mean и log_variance
z_mean = layers.Dense(LATENT_DIM, name='z_mean')(x)
z_log_var = layers.Dense(LATENT_DIM, name='z_log_var')(x)

# Семплирование
z = Sampling()([z_mean, z_log_var])

encoder = keras.Model(encoder_inputs, [z_mean, z_log_var, z], name='encoder')
encoder.summary()

In [ ]:
# === DECODER ===
decoder_inputs = layers.Input(shape=(LATENT_DIM,), name='decoder_input')

x = layers.Dense(7 * 7 * 64, activation='relu')(decoder_inputs)
x = layers.Reshape((7, 7, 64))(x)
x = layers.Conv2DTranspose(64, 3, activation='relu', strides=2, padding='same')(x)
x = layers.Conv2DTranspose(32, 3, activation='relu', strides=2, padding='same')(x)
decoder_outputs = layers.Conv2DTranspose(1, 3, activation='sigmoid', padding='same')(x)

decoder = keras.Model(decoder_inputs, decoder_outputs, name='decoder')
decoder.summary()

In [ ]:
# === VAE Model ===
class VAE(keras.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.total_loss_tracker = keras.metrics.Mean(name='total_loss')
        self.reconstruction_loss_tracker = keras.metrics.Mean(name='reconstruction_loss')
        self.kl_loss_tracker = keras.metrics.Mean(name='kl_loss')
    
    @property
    def metrics(self):
        return [
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker
        ]
    
    def train_step(self, data):
        with tf.GradientTape() as tape:
            # Encode
            z_mean, z_log_var, z = self.encoder(data)
            # Decode
            reconstruction = self.decoder(z)
            
            # Reconstruction loss (как хорошо восстановили)
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(
                    keras.losses.binary_crossentropy(data, reconstruction),
                    axis=(1, 2)
                )
            )
            
            # KL Divergence (насколько распределение близко к N(0,1))
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1
                )
            )
            
            total_loss = reconstruction_loss + kl_loss
        
        # Обновление весов
        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        # Метрики
        self.total_loss_tracker.update_state(total_loss)
        self.reconstruction_loss_tracker.update_state(reconstruction_loss)
        self.kl_loss_tracker.update_state(kl_loss)
        
        return {
            'total_loss': self.total_loss_tracker.result(),
            'reconstruction_loss': self.reconstruction_loss_tracker.result(),
            'kl_loss': self.kl_loss_tracker.result()
        }

vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
print('VAE создан!')
print(f'Латентное пространство: {LATENT_DIM}D')

## 4. Обучение VAE

In [ ]:
# Обучение
history = vae.fit(
    x_train,
    epochs=20,
    batch_size=128,
    verbose=1
)

In [ ]:
# График обучения
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history.history['total_loss'])
axes[0].set_title('Total Loss')
axes[0].set_xlabel('Эпоха')
axes[0].grid(True)

axes[1].plot(history.history['reconstruction_loss'])
axes[1].set_title('Reconstruction Loss')
axes[1].set_xlabel('Эпоха')
axes[1].grid(True)

axes[2].plot(history.history['kl_loss'])
axes[2].set_title('KL Divergence')
axes[2].set_xlabel('Эпоха')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 5. Визуализация латентного пространства

In [ ]:
# Кодируем тестовые данные
z_mean, z_log_var, z = encoder.predict(x_test, verbose=0)

plt.figure(figsize=(12, 10))
scatter = plt.scatter(z[:, 0], z[:, 1], c=y_test, cmap='tab10', alpha=0.5, s=5)
plt.colorbar(scatter, label='Цифра')
plt.xlabel('z[0]')
plt.ylabel('z[1]')
plt.title('Латентное пространство VAE (2D)')
plt.grid(True, alpha=0.3)
plt.show()

print('Цифры образуют кластеры в латентном пространстве!')
print('Пространство непрерывно — можно семплировать из любой точки')

## 6. Генерация новых изображений

In [ ]:
# Генерируем сетку изображений из латентного пространства
n = 15  # размер сетки
figure = np.zeros((28 * n, 28 * n))

# Диапазон латентного пространства (примерно -3 до 3)
grid_x = np.linspace(-3, 3, n)
grid_y = np.linspace(-3, 3, n)[::-1]

for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])
        x_decoded = decoder.predict(z_sample, verbose=0)
        digit = x_decoded[0].reshape(28, 28)
        figure[i * 28:(i + 1) * 28, j * 28:(j + 1) * 28] = digit

plt.figure(figsize=(12, 12))
plt.imshow(figure, cmap='gray')
plt.title('Генерация из латентного пространства VAE')
plt.xlabel('z[0]')
plt.ylabel('z[1]')
plt.axis('off')
plt.show()

print('Каждая точка латентного пространства → уникальное изображение!')

## 7. Реконструкция изображений

In [ ]:
# Реконструкция тестовых изображений
z_mean, z_log_var, z = encoder.predict(x_test[:10], verbose=0)
reconstructed = decoder.predict(z, verbose=0)

fig, axes = plt.subplots(2, 10, figsize=(15, 3))

for i in range(10):
    axes[0, i].imshow(x_test[i].reshape(28, 28), cmap='gray')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_title('Оригинал', loc='left')
axes[1, 0].set_title('Реконструкция', loc='left')

plt.suptitle('VAE: Реконструкция изображений')
plt.tight_layout()
plt.show()

## 8. Интерполяция между цифрами

In [ ]:
# Интерполяция между двумя изображениями
def interpolate(img1, img2, steps=10):
    z1, _, _ = encoder.predict(img1.reshape(1, 28, 28, 1), verbose=0)
    z2, _, _ = encoder.predict(img2.reshape(1, 28, 28, 1), verbose=0)
    
    interpolated = []
    for alpha in np.linspace(0, 1, steps):
        z = z1 * (1 - alpha) + z2 * alpha
        img = decoder.predict(z, verbose=0)
        interpolated.append(img.reshape(28, 28))
    
    return interpolated

# Примеры интерполяции
pairs = [(0, 1), (3, 8), (4, 9), (2, 7)]

fig, axes = plt.subplots(len(pairs), 10, figsize=(15, 6))

for row, (idx1, idx2) in enumerate(pairs):
    # Находим примеры этих цифр
    img1 = x_test[y_test == idx1][0]
    img2 = x_test[y_test == idx2][0]
    
    interp = interpolate(img1, img2)
    
    for col, img in enumerate(interp):
        axes[row, col].imshow(img, cmap='gray')
        axes[row, col].axis('off')
    
    axes[row, 0].set_ylabel(f'{idx1}→{idx2}')

plt.suptitle('Интерполяция между цифрами в латентном пространстве')
plt.tight_layout()
plt.show()

print('Плавный переход между цифрами!')

## 9. Случайная генерация

In [ ]:
# Генерируем случайные изображения из N(0, 1)
n_samples = 25
random_z = np.random.normal(size=(n_samples, LATENT_DIM))
generated = decoder.predict(random_z, verbose=0)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated[i].reshape(28, 28), cmap='gray')
    ax.axis('off')

plt.suptitle('Случайно сгенерированные цифры из N(0,1)', fontsize=14)
plt.tight_layout()
plt.show()

print('Эти цифры не существуют в датасете — они полностью новые!')

## 10. Арифметика в латентном пространстве

In [ ]:
# Вычисляем средние z для каждой цифры
digit_z_means = {}
for digit in range(10):
    mask = y_test == digit
    digit_images = x_test[mask][:100]  # берём 100 примеров
    z_mean_digit, _, _ = encoder.predict(digit_images, verbose=0)
    digit_z_means[digit] = z_mean_digit.mean(axis=0)

# Визуализация средних точек
plt.figure(figsize=(10, 8))
for digit, z in digit_z_means.items():
    plt.scatter(z[0], z[1], s=200, marker=f'${digit}$', c='red')

# Фон — все точки
z_all, _, _ = encoder.predict(x_test[:2000], verbose=0)
plt.scatter(z_all[:, 0], z_all[:, 1], c=y_test[:2000], cmap='tab10', alpha=0.2, s=5)

plt.xlabel('z[0]')
plt.ylabel('z[1]')
plt.title('Средние точки для каждой цифры')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Арифметика: что будет если сложить/вычесть цифры?
# Пример: 1 + (7 - 1) ≈ 7

z_1 = digit_z_means[1]
z_7 = digit_z_means[7]
z_result = z_1 + (z_7 - z_1)  # должно быть ≈ 7

# Генерируем изображения
img_1 = decoder.predict(z_1.reshape(1, -1), verbose=0)
img_7 = decoder.predict(z_7.reshape(1, -1), verbose=0)
img_result = decoder.predict(z_result.reshape(1, -1), verbose=0)
img_diff = decoder.predict((z_7 - z_1).reshape(1, -1), verbose=0)

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
titles = ['1', '+', '(7 - 1)', '=', '?']
images = [img_1, None, img_diff, None, img_result]

for i, ax in enumerate(axes):
    if images[i] is not None:
        ax.imshow(images[i].reshape(28, 28), cmap='gray')
    else:
        ax.text(0.5, 0.5, titles[i], fontsize=40, ha='center', va='center')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(titles[i], fontsize=14)

plt.suptitle('Арифметика в латентном пространстве VAE')
plt.tight_layout()
plt.show()

## Итог

```
┌──────────────────────────────────────────────────────────────────────┐
│                              VAE                                     │
├──────────────────────────────────────────────────────────────────────┤
│  Encoder → (μ, σ)    — кодирует в распределение, не в точку          │
│  Sampling z          — семплирование с reparametrization trick       │
│  Decoder             — генерирует из латентного пространства         │
├──────────────────────────────────────────────────────────────────────┤
│  Loss = Reconstruction + KL Divergence                               │
│       = Качество восстановления + Регуляризация распределения        │
├──────────────────────────────────────────────────────────────────────┤
│  Преимущества перед обычным Autoencoder:                             │
│  • Непрерывное латентное пространство                                │
│  • Можно семплировать из любой точки                                 │
│  • Лучше генерация новых данных                                      │
│  • Интерполяция всегда даёт осмысленные результаты                   │
├──────────────────────────────────────────────────────────────────────┤
│  Применения:                                                         │
│  • Генерация изображений                                             │
│  • Аугментация данных                                                │
│  • Поиск аномалий                                                    │
│  • Сжатие данных                                                     │
│  • Основа для более сложных моделей (β-VAE, VQ-VAE)                  │
└──────────────────────────────────────────────────────────────────────┘
```